# 🎨 Paint-Code-RL: Zero-Cost GRPO Generative Art on Kaggle GPU

This notebook lets you **run generative art synthesis** and **train GRPO reinforcement learning policies** directly in your browser on free Kaggle NVIDIA GPUs (Tesla T4 / P100).

### 🚀 Key Features:
1. **Evaluate Trained KaggleHub Checkpoints**: Automatically pull and run `pernavjain/paint-code/pyTorch/default`.
2. **Sandboxed WebGL Rendering**: Headless Chromium + SwiftShader/ANGLE on port 3000.
3. **Multi-Signal Visual RL Rewards**: Syntax, Visual Richness, Brush Utilization, and Anti-Cheat gates.
4. **Interactive Cyclic Training**: Train 25-step cycles with `--max` GPU saturation and live dashboard tracking.
5. **One-Click Download**: Automatically package all rendered artworks and LoRA weights as a downloadable ZIP.

In [ ]:
# Cell 1: Check Hardware & GPU Acceleration
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# Cell 2: Install System Dependencies for Headless WebGL Chromium
!apt-get update -qq
!apt-get install -y -qq chromium-browser nodejs npm xvfb > /dev/null 2>&1
!node -v && npm -v

In [ ]:
# Cell 3: Clone Paint-Code-RL Repository
import os
if not os.path.exists("paint-code-rl"):
    !git clone https://github.com/harshitthek/paint-code-rl.git
%cd paint-code-rl
!git checkout feat/visual-rl-and-cyclic-training || git checkout main
!git pull origin feat/visual-rl-and-cyclic-training || true

In [ ]:
# Cell 4: Install Python RL Dependencies & Renderer Modules
!pip install -q trl==0.15.1 transformers==4.49.0 peft datasets accelerate pydantic safetensors Pillow pyyaml psutil requests kagglehub
%cd renderer
!npm install --silent
%cd ..

In [ ]:
# Cell 5: Start Headless Node.js WebGL Renderer Daemon
import subprocess, time, requests

proc = subprocess.Popen(["node", "renderer/server.js"])
time.sleep(3)

try:
    health = requests.get("http://127.0.0.1:3000/health", timeout=5).json()
    print("[OK] Renderer daemon is live:", health)
except Exception as e:
    print("[ERROR] Renderer failed to start:", e)

In [ ]:
# Cell 6: Generate Artwork Using Your Uploaded KaggleHub Checkpoint
# Downloads pernavjain/paint-code/pyTorch/default and renders high-res artworks
!python scripts/generate_and_render.py --kagglehub pernavjain/paint-code/pyTorch/default --output-dir artifacts/renders

import glob
from IPython.display import display, Image

renders = sorted(glob.glob("artifacts/renders/*.png"))
print(f"Rendered {len(renders)} artworks:")
for r in renders:
    print(f"File: {r}")
    display(Image(filename=r, width=400))

In [ ]:
# Cell 7: (Optional) Run GRPO Cyclic Training on Kaggle GPU
import os
os.environ["ENV"] = "kaggle"
os.environ["PYTHONUNBUFFERED"] = "1"

# Run 25 steps with GPU hardware saturation & live dashboard
!python scripts/train_grpo.py --mode train --steps-per-cycle 25 --max-steps 25 --unattended --max --dashboard

In [ ]:
# Cell 8: View Live Dashboard Inline
from IPython.display import display, HTML
if os.path.exists("artifacts/dashboard.html"):
    with open("artifacts/dashboard.html", "r", encoding="utf-8") as f:
        html_code = f.read()
    display(HTML(f'<iframe srcdoc="{html_code.replace(chr(34), "&quot;")}" width="100%" height="600px" frameborder="0"></iframe>'))

In [ ]:
# Cell 9: Package All Outputs into Downloadable ZIP
# After running this, download 'paint_rl_artifacts.zip' from Kaggle's right-hand Output panel!
!python scripts/package_artifacts.py --output-dir /kaggle/working
print("\n[DONE] Check the right-hand panel under 'Output' to download your ZIP file!")